# BRCA Metabolic Flux Pipeline — 02. Supervised, unsupervised and clinical analysis

Reads the outputs of notebook 01 (`results/pythonResults/processedData`) and of `matlab/BRCA_fluxAnalysis.m` (`results/matlabResults`), and writes every table and figure to `results/pythonResults` (`figures/`, `tables/`).

| Section | Question |
|---|---|
| 1–2 | Load fluxes, model QC and clinical data; build the feature matrix |
| 3 | Supervised: can flux features separate tumour from normal tissue? (nested, patient-grouped CV) |
| 4 | Unsupervised: metabolic clusters among tumours, with **artefact checks** |
| 5 | Clinical clusters and metabolic–clinical concordance (ARI/AMI) |
| 6 | Characterisation of metabolic clusters: effect sizes, clinical associations, survival, subtypes |
| 7 | Pareto surfaces and model size |

Design decisions that differ from the previous analysis (and why):

* **Model QC first.** Models that failed validation in MATLAB (`QC_pass = 0`, e.g. XomicsToModel relaxation, infeasible or zero biomass) are excluded. In the previous version, relaxed models with zero biomass formed part of the "Divergent" group.
* **Patient-grouped, nested cross-validation.** Tumour and normal samples of the same patient always fall in the same fold, and feature selection and hyper-parameter search are done inside each training fold. Reporting the best of many configurations evaluated on the same folds inflates the AUC.
* **Balanced metrics.** Tumours outnumber normals ~10:1, so balanced accuracy and MCC are reported with ROC-AUC; a paired subset (one tumour and one normal per patient) is analysed separately.
* **Clusters are tested against technical covariates** (missing clinical metadata, tissue source site, model QC warnings) before being interpreted biologically.

Optional packages: `umap-learn`, `hdbscan`, `lifelines`. Without them the notebook falls back to PCA, skips HDBSCAN and skips the survival models.

## 0. Configuration

In [ ]:
import os, re, json, sys, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DRIVE_DIR  = Path(os.environ.get("BRCA_DRIVE_DIR",
                                 Path.home() / "work/Drive/GIMI/01_posgrado/MCBCI_analisisCancer"))
PROC_DIR   = DRIVE_DIR / "results" / "pythonResults" / "processedData"   # outputs of notebook 01
MATLAB_DIR = DRIVE_DIR / "results" / "matlabResults"                    # outputs of matlab/BRCA_fluxAnalysis.m
OUT_DIR    = DRIVE_DIR / "results" / "pythonResults"                    # everything this notebook writes

FEATURE_FILE = MATLAB_DIR / "FeatureMatrix_TumorPhenotype_All.csv"
QC_FILE      = MATLAB_DIR / "ModelQC.csv"                 # optional
PARETO_FILE  = MATLAB_DIR / "ParetoSurface_100sol.csv"    # optional

SOL_NAMES = ["FBA", "pFBA", "L1w"]   # flux solutions used as features (MATLAB also writes L2, L2w)
SEEDS     = [42, 123, 100]
FAST      = False                    # True: small sweeps for a quick test (results not for the paper)
N_JOBS    = -1
# >>> CONFIG END

In [ ]:
if FAST:
    SEEDS = SEEDS[:1]
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)
for p in [FEATURE_FILE, QC_FILE, PARETO_FILE, PROC_DIR / "clinical_processed.csv", PROC_DIR / "sample_inventory.csv"]:
    print(("OK      " if p.exists() else "missing ") + str(p))

## 1. Imports and figure style

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, kruskal
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift, estimate_bandwidth
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, ParameterGrid
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, balanced_accuracy_score,
                             matthews_corrcoef, recall_score, f1_score, confusion_matrix, roc_curve,
                             silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, adjusted_mutual_info_score)
from scipy.stats import loguniform, randint
warnings.filterwarnings("ignore")

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
try:
    import hdbscan
    HAVE_HDBSCAN = True
except ImportError:
    HAVE_HDBSCAN = False
try:
    from lifelines import KaplanMeierFitter, CoxPHFitter
    from lifelines.statistics import multivariate_logrank_test
    HAVE_LIFELINES = True
except ImportError:
    HAVE_LIFELINES = False
print(f"umap: {HAVE_UMAP}   hdbscan: {HAVE_HDBSCAN}   lifelines: {HAVE_LIFELINES}")

PALETTE = {"tumor": "#F18F01", "normal": "#3BB273", "neutral": "#A9A9A9"}
CLUSTER_COLORS = ["#2E86AB", "#E84855", "#7B2D8B", "#F7B731", "#3BB273", "#A9A9A9"]
DPI = 300
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 12,
                     "axes.labelsize": 11, "figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close("all")
    print(f"  saved {path.name}")

def bh(p):
    """Benjamini-Hochberg adjusted p-values (NaN-safe)."""
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    n = ok.sum()
    if n == 0:
        return out
    order = np.argsort(p[ok])
    ranked = p[ok][order] * n / np.arange(1, n + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1].clip(max=1)
    tmp = np.empty(n); tmp[order] = ranked
    out[ok] = tmp
    return out

def cliffs_delta(x, y):
    """Cliff's delta from the Mann-Whitney U statistic: P(x>y) - P(x<y)."""
    u = mannwhitneyu(x, y, alternative="two-sided")
    return 2 * u.statistic / (len(x) * len(y)) - 1, u.pvalue

def cramers_v(a, b):
    t = pd.crosstab(a, b)
    if t.shape[0] < 2 or t.shape[1] < 2:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(t)
    v = np.sqrt(chi2 / (t.to_numpy().sum() * (min(t.shape) - 1)))
    return v, p

def to_sample_id(s):
    """'TCGA-3C-AAAU-01A_Xomics_specificModel', 'TCGA_3C_AAAU_01A_Xomics' -> 'TCGA-3C-AAAU-01A'."""
    m = re.search(r"TCGA[-_.][A-Z0-9]{2}[-_.][A-Z0-9]{4}[-_.]\d{2}[A-Z]?", str(s))
    return re.sub(r"[_.]", "-", m.group(0))[:16] if m else str(s)

## 2. Data loading and feature matrix

In [ ]:
feat = pd.read_csv(FEATURE_FILE, low_memory=False)
feat["sample"] = feat["Model"].map(to_sample_id)
assert feat["sample"].is_unique, "two models map to the same sample"
inv  = pd.read_csv(PROC_DIR / "sample_inventory.csv")
clin = pd.read_csv(PROC_DIR / "clinical_processed.csv")
print(f"models in feature matrix: {len(feat):,}   columns: {feat.shape[1]:,}")
print(f"models not in the sample inventory: {(~feat['sample'].isin(inv['sample'])).sum()}")

# ---- model QC (written by matlab/BRCA_fluxAnalysis.m); old matrices without QC are treated as passing
qc = pd.DataFrame({"sample": feat["sample"]})
if QC_FILE.exists():
    q = pd.read_csv(QC_FILE)
    q["sample"] = q["Model"].map(to_sample_id)
    qc = qc.merge(q.drop(columns="Model"), on="sample", how="left")
if "QC_pass" in feat:
    qc["QC_pass"] = feat["QC_pass"].fillna(0).astype(int).to_numpy()
else:
    qc["QC_pass"] = 1
    print("WARNING: no QC_pass column (feature matrix from the old pipeline); all models kept")
qc = qc.merge(inv[["sample", "sample_type", "patient", "tss_code", "in_metadata", "is_first_tumor_vial"]],
              on="sample", how="left")
qc.to_csv(TAB_DIR / "model_qc_merged.csv", index=False)
print("\nQC by sample type:")
print(pd.crosstab(qc["sample_type"], qc["QC_pass"].map({1: "pass", 0: "fail"}), margins=True))
if "Status" in qc:
    print("\nQC status:", qc["Status"].value_counts().to_dict())
    print(qc.loc[qc.QC_pass == 0, "FailReasons"].str.split(" | ", regex=False).explode().value_counts().head(10))

In [ ]:
# ---- feature groups (secondary metabolic metrics: same definitions as the paper)
METRIC_ROOTS = ["CU", "EA", "WarburgIndex", "ATPConsumption", "ATPProduction", "RedoxIndex", "MFI",
                "AnabolismScore", "NADPHdemand", "TCA_completeness", "LipidSat", "LipidUnsat",
                "LipidPL", "GlnDependence"]
cols = set(feat.columns)
metric_cols = [f"{r}_{s}" for r in METRIC_ROOTS for s in SOL_NAMES if f"{r}_{s}" in cols]
sa_cols     = sorted(c for c in cols if c.startswith("SA_") and c.rsplit("_", 1)[-1] in SOL_NAMES)
onc_cols    = sorted(c for c in cols if c.startswith("Oncomet_") and c.rsplit("_", 1)[-1] in SOL_NAMES)
flux_cols   = sorted(c for c in cols if c.startswith("Flux_") and c.rsplit("_", 1)[-1] in SOL_NAMES)
size_cols   = [c for c in ["nRxns", "nMets", "nGenes", "MaxBiomass_FBA"] if c in cols]
FEATURES = metric_cols + sa_cols + onc_cols
print(f"metrics: {len(metric_cols)}  subsystem activities: {len(sa_cols)}  oncometabolites: {len(onc_cols)}"
      f"  -> {len(FEATURES)} features   (raw flux columns not used: {len(flux_cols):,})")

# ---- analysis table: QC-passing models with their clinical data
df = (feat[["sample"] + FEATURES + size_cols]
      .merge(qc[["sample", "QC_pass"]], on="sample")
      .merge(clin, on="sample", how="left")
      .merge(inv[["sample", "is_first_tumor_vial", "paired_tumor_normal"]], on="sample", how="left"))
df = df[df["QC_pass"] == 1].reset_index(drop=True)

# unsupervised column filter (no labels used): drop features that are >=95% zero/NaN or constant
Xall = df[FEATURES].replace([np.inf, -np.inf], np.nan)
bad = ((Xall.fillna(0) == 0).mean() >= 0.95) | (Xall.std() < 1e-10)
FEATURES = [f for f in FEATURES if not bad[f]]
print(f"features after removing near-empty/constant columns: {len(FEATURES)}")
print(f"QC-passing models: {len(df):,}  ->", df["sample_type"].value_counts().to_dict())

## 3. Supervised learning: tumour vs normal

Two datasets:
* **all** — every QC-passing tumour and normal sample (≈10 tumours per normal);
* **paired** — patients with both tissues: their first tumour vial and their normal sample (balanced, each patient is its own control).

Each classifier is a pipeline `median imputation → Mann-Whitney/BH feature selection → scaling → classifier`, so every step is learnt on training folds only.
Outer loop: 5-fold `StratifiedGroupKFold` by patient × seeds. Inner loop: `RandomizedSearchCV` (3-fold, grouped) on the outer-training data only. All metrics come from outer-fold predictions.

In [ ]:
class MWUSelector(BaseEstimator, TransformerMixin):
    """Keep features with BH-adjusted Mann-Whitney p <= alpha (at least k_min, the smallest p-values)."""
    def __init__(self, alpha=0.05, k_min=10):
        self.alpha, self.k_min = alpha, k_min
    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        p = mannwhitneyu(X[y == 1], X[y == 0], axis=0, alternative="two-sided").pvalue
        p = np.where(np.isnan(p), 1.0, p)
        keep = bh(p) <= self.alpha
        if keep.sum() < self.k_min:
            keep = np.zeros_like(keep); keep[np.argsort(p)[: self.k_min]] = True
        self.support_ = keep
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.support_]

N_ITER = 4 if FAST else 20
SPACES = {
    "Dummy (prior)": (DummyClassifier(strategy="prior"), {}),
    "Logistic Regression": (LogisticRegression(max_iter=5000, class_weight="balanced"),
                            {"clf__C": loguniform(1e-3, 10)}),
    "SVM": (SVC(probability=True, class_weight="balanced"),
            {"clf__C": loguniform(1e-2, 100), "clf__kernel": ["rbf", "linear"], "clf__gamma": ["scale", 1e-3, 1e-2, 1e-1]}),
    "KNN": (KNeighborsClassifier(),
            {"clf__n_neighbors": randint(3, 31), "clf__weights": ["uniform", "distance"],
             "clf__metric": ["euclidean", "manhattan", "cosine"]}),
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced"),
                      {"clf__max_depth": randint(2, 12), "clf__min_samples_leaf": randint(1, 20)}),
    "Random Forest": (RandomForestClassifier(class_weight="balanced", n_jobs=1),
                      {"clf__n_estimators": randint(100, 400), "clf__max_depth": randint(3, 16),
                       "clf__max_features": ["sqrt", "log2", 0.3], "clf__min_samples_leaf": randint(1, 10)}),
    "Naive Bayes": (GaussianNB(), {"clf__var_smoothing": loguniform(1e-12, 1e-6)}),
}

def make_pipe(clf):
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("select", MWUSelector()),
                     ("scale", StandardScaler()), ("clf", clf)])

def nested_cv(X, y, groups, label):
    rows, oof, params = [], [], []
    for seed in SEEDS:
        outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold, (tr, te) in enumerate(outer.split(X, y, groups)):
            for name, (clf, space) in SPACES.items():
                pipe = make_pipe(clone(clf))
                if hasattr(pipe.named_steps["clf"], "random_state"):
                    pipe.set_params(clf__random_state=seed)
                if space:
                    inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
                    model = RandomizedSearchCV(pipe, space, n_iter=N_ITER, cv=inner, scoring="roc_auc",
                                               random_state=seed, n_jobs=N_JOBS, error_score=np.nan)
                    model.fit(X[tr], y[tr], groups=groups[tr])
                    params.append({"dataset": label, "model": name, "seed": seed, "fold": fold,
                                   **{k.replace("clf__", ""): v for k, v in model.best_params_.items()}})
                    n_sel = model.best_estimator_.named_steps["select"].support_.sum()
                else:
                    model = pipe.fit(X[tr], y[tr]); n_sel = model.named_steps["select"].support_.sum()
                prob = model.predict_proba(X[te])[:, 1]
                pred = (prob >= 0.5).astype(int)
                rows.append({"dataset": label, "model": name, "seed": seed, "fold": fold, "n_features": n_sel,
                             "ROC_AUC": roc_auc_score(y[te], prob), "PR_AUC": average_precision_score(y[te], prob),
                             "balanced_accuracy": balanced_accuracy_score(y[te], pred),
                             "MCC": matthews_corrcoef(y[te], pred), "sensitivity": recall_score(y[te], pred),
                             "specificity": recall_score(y[te], pred, pos_label=0), "F1": f1_score(y[te], pred)})
                oof.append(pd.DataFrame({"dataset": label, "model": name, "seed": seed, "idx": te,
                                         "y": y[te], "prob": prob}))
        print(f"  {label}: seed {seed} done")
    return pd.DataFrame(rows), pd.concat(oof), pd.DataFrame(params)

lab = df[df.sample_type.isin(["Tumor", "Normal"])].reset_index(drop=True)
paired_pat = set(lab.loc[lab.sample_type == "Normal", "patient"]) & set(lab.loc[lab.sample_type == "Tumor", "patient"])
DATASETS = {
    "all": lab,
    "paired": lab[lab.patient.isin(paired_pat) & ((lab.sample_type == "Normal") | lab.is_first_tumor_vial)].reset_index(drop=True),
}
cv_rows, cv_oof, cv_params = [], [], []
for label, d in DATASETS.items():
    X = d[FEATURES].to_numpy(float); y = (d.sample_type == "Tumor").astype(int).to_numpy()
    g = d.patient.to_numpy()
    print(f"{label}: {len(d)} samples ({y.sum()} tumour / {(1-y).sum()} normal), {len(set(g))} patients")
    r, o, p = nested_cv(X, y, g, label)
    cv_rows.append(r); cv_oof.append(o); cv_params.append(p)
cv_rows, cv_oof, cv_params = pd.concat(cv_rows), pd.concat(cv_oof), pd.concat(cv_params)
cv_rows.to_csv(TAB_DIR / "supervised_nestedCV_folds.csv", index=False)
cv_oof.to_csv(TAB_DIR / "supervised_nestedCV_predictions.csv", index=False)
cv_params.to_csv(TAB_DIR / "supervised_nestedCV_best_params.csv", index=False)

METRICS = ["ROC_AUC", "PR_AUC", "balanced_accuracy", "MCC", "sensitivity", "specificity", "F1"]
summary = cv_rows.groupby(["dataset", "model"])[METRICS + ["n_features"]].agg(["mean", "std"])
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
summary = summary.reset_index().sort_values(["dataset", "ROC_AUC_mean"], ascending=[True, False])
summary.to_csv(TAB_DIR / "supervised_nestedCV_summary.csv", index=False)
summary[["dataset", "model"] + [f"{m}_mean" for m in METRICS]].round(3)

In [ ]:
# ---- figures: metric heatmap, out-of-fold ROC curves, confusion matrix of the best model
for label in DATASETS:
    s = summary[summary.dataset == label].set_index("model")
    m = s[[f"{k}_mean" for k in METRICS]]; m.columns = METRICS
    sd = s[[f"{k}_std" for k in METRICS]].to_numpy()
    annot = np.array([[f"{v:.2f}\n±{e:.2f}" for v, e in zip(r, se)] for r, se in zip(m.to_numpy(), sd)])
    fig, ax = plt.subplots(figsize=(10, 0.6 * len(m) + 1.5))
    sns.heatmap(m, annot=annot, fmt="", cmap="YlGnBu", vmin=0, vmax=1, cbar_kws={"label": "mean (outer folds)"}, ax=ax)
    ax.set_title(f"Tumour vs normal — nested patient-grouped CV ({label})"); ax.set_ylabel("")
    savefig(f"supervised_metrics_heatmap_{label}.png")

    fig, ax = plt.subplots(figsize=(6, 5.5))
    for i, name in enumerate(s.index):
        o = cv_oof[(cv_oof.dataset == label) & (cv_oof.model == name) & (cv_oof.seed == SEEDS[0])]
        fpr, tpr, _ = roc_curve(o.y, o.prob)
        ax.plot(fpr, tpr, lw=1.6, label=f"{name} ({roc_auc_score(o.y, o.prob):.2f})",
                color=(CLUSTER_COLORS + ["black", "grey"])[i % 8])
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"Out-of-fold ROC, seed {SEEDS[0]} ({label})"); ax.legend(fontsize=8, loc="lower right")
    savefig(f"supervised_roc_{label}.png")

    best = s.drop(index="Dummy (prior)").index[0]
    o = cv_oof[(cv_oof.dataset == label) & (cv_oof.model == best) & (cv_oof.seed == SEEDS[0])]
    cm = confusion_matrix(o.y, (o.prob >= 0.5).astype(int))
    fig, ax = plt.subplots(figsize=(4, 3.6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Normal", "Tumour"], yticklabels=["Normal", "Tumour"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{best} ({label})")
    savefig(f"supervised_confusion_{label}.png")

## 4. Unsupervised metabolic clustering (tumours)

One tumour per patient (first vial), QC-passing models only. Embeddings: UMAP grid (if installed) and PCA; algorithms: K-means, agglomerative, GMM, Bayesian GMM, DBSCAN, mean-shift, HDBSCAN.
Silhouette computed in a 2–3-D UMAP space is optimistic, so the selected solutions are also scored in the original (scaled) feature space.

In [ ]:
tum = df[(df.sample_type == "Tumor") & df.is_first_tumor_vial].reset_index(drop=True)
X_tum = RobustScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(tum[FEATURES]))
print(f"tumours for clustering: {len(tum)}   features: {X_tum.shape[1]}")

def embeddings(X, seeds, fast=FAST):
    E = {}
    for nc in (2, 3):
        E[f"PCA_C{nc}"] = PCA(n_components=nc, random_state=0).fit_transform(X)
    if HAVE_UMAP:
        grid = ParameterGrid({"n_neighbors": [15, 30] if fast else [5, 15, 30, 50],
                              "min_dist": [0.1] if fast else [0.0, 0.1, 0.5],
                              "n_components": [2, 3], "metric": ["euclidean"] if fast else ["euclidean", "cosine"]})
        for seed in seeds:
            for p in grid:
                key = f"UMAP_N{p['n_neighbors']}_D{p['min_dist']}_C{p['n_components']}_{p['metric']}_S{seed}"
                E[key] = umap.UMAP(random_state=seed, n_jobs=1, **p).fit_transform(X)
    return E

def algorithms(Xref):
    bw = estimate_bandwidth(Xref, quantile=0.2, n_samples=min(500, len(Xref)), random_state=0) or 1.0
    A = {"KMeans": (KMeans, ParameterGrid({"n_clusters": range(2, 7), "n_init": [10]}), True),
         "Agglomerative": (AgglomerativeClustering, ParameterGrid({"n_clusters": range(2, 7), "linkage": ["ward", "average"]}), False),
         "GMM": (GaussianMixture, ParameterGrid({"n_components": range(2, 6)}), True),
         "BayesGMM": (BayesianGaussianMixture, ParameterGrid({"n_components": range(2, 6)}), True),
         "DBSCAN": (DBSCAN, ParameterGrid({"eps": [0.3, 0.5, 1.0, 1.5], "min_samples": [5, 10]}), False),
         "MeanShift": (MeanShift, ParameterGrid({"bandwidth": [bw * 0.5, bw, bw * 1.5]}), False)}
    if HAVE_HDBSCAN:
        A["HDBSCAN"] = (hdbscan.HDBSCAN, ParameterGrid({"min_cluster_size": [10, 20, 40]}), False)
    return A

def clustering_sweep(E, seeds, max_noise=5.0, min_cluster=10):
    rows = []
    for ek, Xe in E.items():
        for an, (cls, grid, stochastic) in algorithms(Xe).items():
            best = None
            for seed in (seeds if stochastic else seeds[:1]):
                for p in grid:
                    try:
                        m = cls(**p, random_state=seed) if stochastic else cls(**p)
                        lab_ = np.asarray(m.fit_predict(Xe))
                    except Exception:
                        continue
                    ok = lab_ != -1
                    sizes = pd.Series(lab_[ok]).value_counts()
                    if 100 * (~ok).mean() > max_noise or len(sizes) < 2 or sizes.min() < min_cluster:
                        continue
                    sil = silhouette_score(Xe[ok], lab_[ok])
                    if best is None or sil > best["silhouette_embedding"]:
                        best = {"embedding": ek, "algorithm": an, "params": str(p), "seed": seed, "k": len(sizes),
                                "noise_pct": 100 * (~ok).mean(), "smallest_cluster": int(sizes.min()),
                                "silhouette_embedding": sil, "davies_bouldin": davies_bouldin_score(Xe[ok], lab_[ok]),
                                "calinski_harabasz": calinski_harabasz_score(Xe[ok], lab_[ok]), "labels": lab_}
            if best:
                rows.append(best)
    return pd.DataFrame(rows).sort_values("silhouette_embedding", ascending=False).reset_index(drop=True)

E_met = embeddings(X_tum, SEEDS)
print(f"embeddings: {len(E_met)}")
res_met = clustering_sweep(E_met, SEEDS)
res_met["silhouette_features"] = [silhouette_score(X_tum[l != -1], l[l != -1]) for l in res_met["labels"]]
res_met.drop(columns="labels").to_csv(TAB_DIR / "clustering_metabolic_sweep.csv", index=False)
res_met.drop(columns="labels").head(10).round(3)

**Artefact check.** A metabolic cluster is only interpreted biologically if it is not explained by technical covariates. For the top solutions we test association (Cramér's V, χ²) with: missing MetaData, tissue source site, MATLAB QC warnings/relaxation, and model size / maximal biomass (Kruskal–Wallis). Flags: V ≥ 0.3 or p < 0.001.

In [ ]:
qc_tum = qc.set_index("sample").reindex(tum["sample"])
def artefact_table(labels, frame):
    ok = labels != -1
    lab_s = pd.Series(labels[ok])
    out = []
    covs = {"missing_metadata": ~frame["in_metadata"].fillna(False).astype(bool),
            "tissue_source_site": frame["tss_code"]}
    if "Warnings" in qc_tum:
        covs["qc_warning"] = qc_tum["Warnings"].fillna("").ne("").to_numpy()
    if "RelaxationUsed" in qc_tum:
        covs["relaxation_used"] = qc_tum["RelaxationUsed"].fillna(0).astype(float).gt(0).to_numpy()
    for k, v in covs.items():
        v_, p_ = cramers_v(lab_s.to_numpy(), np.asarray(v)[ok])
        out.append({"covariate": k, "test": "chi2", "cramers_v": v_, "p": p_})
    for k in [c for c in size_cols if c in frame]:
        x = frame[k].to_numpy()[ok]
        grp = [x[lab_s.to_numpy() == c] for c in lab_s.unique()]
        out.append({"covariate": k, "test": "kruskal", "cramers_v": np.nan, "p": kruskal(*grp).pvalue})
    t = pd.DataFrame(out)
    t["flag"] = (t.cramers_v >= 0.3) | (t.p < 1e-3)
    return t

TOP_N = min(5, len(res_met))
art = []
for i in range(TOP_N):
    t = artefact_table(res_met.loc[i, "labels"], tum)
    t.insert(0, "solution", i); art.append(t)
art = pd.concat(art); art.to_csv(TAB_DIR / "clustering_metabolic_artefact_checks.csv", index=False)
flagged = art[art.flag]
print("Solutions with a technical association:" if len(flagged) else "No technical association in the top solutions.")
art.pivot(index="covariate", columns="solution", values="p").map(lambda v: f"{v:.1e}")

In [ ]:
# ---- selected metabolic solution: best silhouette among those without artefact flags
clean = [i for i in range(TOP_N) if not art[(art.solution == i)].flag.any()]
SEL = clean[0] if clean else 0
if not clean:
    print("WARNING: every top solution is associated with a technical covariate; interpret with caution")
sel = res_met.loc[SEL]
tum["metabolic_cluster"] = sel["labels"]
print(f"selected: {sel.embedding} | {sel.algorithm} {sel.params} | k={sel.k} | "
      f"silhouette embedding {sel.silhouette_embedding:.3f}, features {sel.silhouette_features:.3f}")
print(tum["metabolic_cluster"].value_counts().to_dict())
tum[["sample", "metabolic_cluster"]].to_csv(TAB_DIR / "metabolic_clusters_selected.csv", index=False)

Xe = E_met[sel.embedding]
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d" if Xe.shape[1] == 3 else None)
for i, c in enumerate(sorted(tum.metabolic_cluster.unique())):
    m = tum.metabolic_cluster.to_numpy() == c
    ax.scatter(*Xe[m].T, s=8, alpha=0.7, color=CLUSTER_COLORS[i % 6] if c != -1 else "lightgrey",
               label=f"{'noise' if c == -1 else f'cluster {c}'} (n={m.sum()})")
ax.set_title(f"Metabolic clusters — {sel.algorithm}, {sel.embedding}"); ax.legend(fontsize=8)
savefig("clusters_metabolic_selected.png")

## 5. Clinical clustering and metabolic–clinical concordance

In [ ]:
CLIN_CAT = ["stage", "T", "N", "M", "histology", "ER", "PR", "HER2", "receptor_subtype", "PAM50",
            "menopausal_status", "genetic_ancestry"]
CLIN_NUM = ["age_years"]
# only tumours with MetaData: otherwise the "missing" categories dominate the clusters
tc = tum[tum.in_metadata.fillna(False).astype(bool)].reset_index(drop=True)
ct = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), CLIN_NUM),
                        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CLIN_CAT)])
X_clin = ct.fit_transform(tc[CLIN_NUM + CLIN_CAT].astype({c: str for c in CLIN_CAT}).replace("nan", np.nan))
print(f"clinical matrix: {X_clin.shape}")
E_clin = embeddings(X_clin, SEEDS)
res_clin = clustering_sweep(E_clin, SEEDS)
res_clin.drop(columns="labels").to_csv(TAB_DIR / "clustering_clinical_sweep.csv", index=False)

# ---- ARI / AMI for every (clinical, metabolic) pair on the common tumours
pos = tum.reset_index().set_index("sample").loc[tc["sample"], "index"].to_numpy()
pairs = []
for i, rc in res_clin.iterrows():
    for j, rm in res_met.iterrows():
        a, b = rc["labels"], rm["labels"][pos]
        ok = (a != -1) & (b != -1)
        pairs.append({"clinical": i, "metabolic": j, "n": ok.sum(), "ARI": adjusted_rand_score(a[ok], b[ok]),
                      "AMI": adjusted_mutual_info_score(a[ok], b[ok])})
ari = pd.DataFrame(pairs).sort_values("ARI", ascending=False)
ari.to_csv(TAB_DIR / "concordance_ARI_AMI.csv", index=False)
print(f"pairs: {len(ari)}   ARI median {ari.ARI.median():.3f}, max {ari.ARI.max():.3f}")
top = ari.head(10).copy()
top["label"] = [f"C{r.clinical}:{res_clin.loc[r.clinical,'algorithm']} × M{r.metabolic}:{res_met.loc[r.metabolic,'algorithm']}"
                for r in top.itertuples()]
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(top.set_index("label")[["ARI", "AMI"]], annot=True, fmt=".3f", cmap="mako_r", ax=ax)
ax.set_title("Top-10 clinical × metabolic concordance"); ax.set_ylabel("")
savefig("concordance_top10.png")

## 6. Characterisation of the selected metabolic clusters

In [ ]:
# ---- effect sizes: each cluster vs the rest (Cliff's delta, Mann-Whitney, BH)
rows = []
lab_ = tum["metabolic_cluster"].to_numpy()
Xraw = tum[FEATURES].to_numpy(float)
for c in sorted(set(lab_) - {-1}):
    a, b = Xraw[lab_ == c], Xraw[(lab_ != c) & (lab_ != -1)]
    for j, f in enumerate(FEATURES):
        x, y_ = a[:, j][~np.isnan(a[:, j])], b[:, j][~np.isnan(b[:, j])]
        if len(x) < 3 or len(y_) < 3:
            continue
        d, p = cliffs_delta(x, y_)
        rows.append({"cluster": c, "feature": f, "n_cluster": len(x), "n_rest": len(y_),
                     "median_cluster": np.median(x), "median_rest": np.median(y_), "cliffs_delta": d, "p": p})
eff = pd.DataFrame(rows)
eff["p_adj"] = eff.groupby("cluster")["p"].transform(bh)
eff["magnitude"] = pd.cut(eff.cliffs_delta.abs(), [-1, 0.147, 0.33, 0.474, 1], labels=["negligible", "small", "medium", "large"])
eff.to_csv(TAB_DIR / "metabolic_clusters_effect_sizes.csv", index=False)

small = tum.metabolic_cluster[tum.metabolic_cluster != -1].value_counts().idxmin()
e = eff[(eff.cluster == small) & (eff.p_adj < 0.05)].reindex(
    eff[(eff.cluster == small) & (eff.p_adj < 0.05)].cliffs_delta.abs().sort_values(ascending=False).index).head(40)
if len(e):
    fig, ax = plt.subplots(figsize=(6, 0.22 * len(e) + 1.2))
    e = e.sort_values("cliffs_delta")
    ax.barh(e.feature.str.replace("_", " "), e.cliffs_delta, color=np.where(e.cliffs_delta < 0, "#2E86AB", "#E84855"))
    for t in (0.147, 0.33, 0.474):
        ax.axvline(t, ls=":", c="grey", lw=0.8); ax.axvline(-t, ls=":", c="grey", lw=0.8)
    ax.set_xlabel("Cliff's delta"); ax.set_title(f"Cluster {small} vs rest (FDR < 0.05, top 40)")
    ax.tick_params(axis="y", labelsize=7)
    savefig("clusters_cliffs_delta_smallest.png")
print(eff[eff.p_adj < 0.05].groupby(["cluster", "magnitude"], observed=True).size().unstack(fill_value=0))

In [ ]:
# ---- clinical associations of the metabolic clusters
rows = []
ok = tum.metabolic_cluster != -1
for v in CLIN_CAT + ["sample_type", "tss_code", "race"]:
    if v not in tum:
        continue
    s = tum.loc[ok & tum[v].notna()]
    cv_, p = cramers_v(s.metabolic_cluster, s[v])
    rows.append({"variable": v, "test": "chi2", "n": len(s), "cramers_v": cv_, "p": p})
for v in CLIN_NUM:
    s = tum.loc[ok & tum[v].notna()]
    rows.append({"variable": v, "test": "kruskal", "n": len(s), "cramers_v": np.nan,
                 "p": kruskal(*[g[v] for _, g in s.groupby("metabolic_cluster")]).pvalue})
assoc = pd.DataFrame(rows); assoc["p_adj"] = bh(assoc.p)
assoc.sort_values("p").to_csv(TAB_DIR / "metabolic_clusters_clinical_associations.csv", index=False)
assoc.sort_values("p").round(4)

In [ ]:
# ---- survival by metabolic cluster
if HAVE_LIFELINES:
    s = tum[(tum.metabolic_cluster != -1) & tum.OS_days.notna() & tum.OS_event.notna()].copy()
    s["OS_years"] = s.OS_days / 365.25
    lr = multivariate_logrank_test(s.OS_years, s.metabolic_cluster, s.OS_event.astype(int))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for i, (c, g) in enumerate(s.groupby("metabolic_cluster")):
        KaplanMeierFitter(label=f"cluster {c} (n={len(g)})").fit(g.OS_years, g.OS_event.astype(int)).plot_survival_function(
            ax=ax, ci_show=True, color=CLUSTER_COLORS[i % 6])
    ax.set_xlabel("Years"); ax.set_ylabel("Overall survival"); ax.set_ylim(0, 1.05)
    ax.set_title(f"Overall survival by metabolic cluster (log-rank p = {lr.p_value:.3g})")
    savefig("survival_km_metabolic_clusters.png")
    # Cox model adjusted for age and stage (complete cases)
    cx = s[["OS_years", "OS_event", "age_years", "stage", "metabolic_cluster"]].dropna()
    cx = pd.get_dummies(cx.astype({"stage": str, "metabolic_cluster": str}), columns=["stage", "metabolic_cluster"],
                        drop_first=True, dtype=float)
    cph = CoxPHFitter().fit(cx, "OS_years", "OS_event")
    cph.summary.to_csv(TAB_DIR / "survival_cox_metabolic_clusters.csv")
    display(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3))
else:
    print("lifelines not installed: survival analysis skipped (pip install lifelines)")

In [ ]:
# ---- metabolic signatures by PAM50 subtype x cluster (median z-score of the top features)
topf = (eff[eff.p_adj < 0.05].assign(a=lambda x: x.cliffs_delta.abs()).sort_values("a", ascending=False)
        .drop_duplicates("feature").feature.head(30).tolist()) or FEATURES[:30]
Z = pd.DataFrame(X_tum, columns=FEATURES)[topf]
Z["group"] = "C" + tum.metabolic_cluster.astype(str) + " | " + tum.PAM50.fillna("NA")
counts = Z.group.value_counts()
H = Z[Z.group.isin(counts[counts >= 5].index)].groupby("group").median()
H = (H - H.mean()) / H.std().replace(0, 1)
fig, ax = plt.subplots(figsize=(11, 0.35 * len(H) + 2))
sns.heatmap(H, cmap="RdBu_r", center=0, vmin=-2, vmax=2, ax=ax, cbar_kws={"label": "z (median per group)"},
            xticklabels=[c.replace("_", " ") for c in H.columns])
ax.tick_params(axis="x", labelsize=7, rotation=90); ax.set_ylabel("cluster | PAM50 (n ≥ 5)")
ax.set_title("Metabolic signatures across clusters and PAM50 subtypes")
savefig("heatmap_clusters_pam50.png")

## 7. Pareto surfaces and model size

In [ ]:
if PARETO_FILE.exists():
    par = pd.read_csv(PARETO_FILE)
    par["sample"] = par["Model"].map(to_sample_id)
    par = par.merge(tum[["sample", "metabolic_cluster"]], on="sample")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, yv in zip(axes, ["CU", "EA"]):
        for i, (c, g) in enumerate(par.groupby("metabolic_cluster")):
            m = g.groupby(pd.cut(g.Biomass, 10), observed=True)[yv].median()
            ax.plot([iv.mid for iv in m.index], m.to_numpy(), marker="o", color=CLUSTER_COLORS[i % 6], label=f"cluster {c}")
        ax.set_xlabel("Biomass flux"); ax.set_ylabel(f"{yv} (median)"); ax.legend(fontsize=8)
    fig.suptitle("Pareto surfaces by metabolic cluster")
    savefig("pareto_by_cluster.png")
else:
    print("Pareto file not found: section skipped")

if size_cols:
    fig, axes = plt.subplots(1, len(size_cols), figsize=(3.2 * len(size_cols), 3.5))
    for ax, c in zip(np.atleast_1d(axes), size_cols):
        sns.boxplot(data=df, x="sample_type", y=c, ax=ax, palette="Set2", fliersize=1)
        ax.set_xlabel("")
    fig.suptitle("Model size and maximal biomass by sample type")
    savefig("model_size_by_sample_type.png")
    df.groupby("sample_type")[size_cols].describe().T.to_csv(TAB_DIR / "model_size_summary.csv")

## 8. Outputs

In [ ]:
log = {"notebook": "BRCA_Metabolic_Flux_Pipeline_02.ipynb", "run_at": datetime.datetime.now().isoformat(timespec="seconds"),
       "feature_file": str(FEATURE_FILE.name), "n_models": int(len(feat)), "n_qc_pass": int(len(df)),
       "solutions": SOL_NAMES, "n_features": len(FEATURES), "seeds": SEEDS, "fast": FAST,
       "packages": {"umap": HAVE_UMAP, "hdbscan": HAVE_HDBSCAN, "lifelines": HAVE_LIFELINES},
       "selected_metabolic_solution": {k: (v if isinstance(v, (int, float, str)) else str(v))
                                       for k, v in sel.drop("labels").items()}}
json.dump(log, open(OUT_DIR / "run_log_02.json", "w"), indent=1, default=str)
for d in (FIG_DIR, TAB_DIR):
    print(f"\n{d}:")
    for f in sorted(d.iterdir()):
        print(f"  {f.name:55s} {f.stat().st_size/1024:8.1f} KB")